In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv('heart_disease.csv', na_values='?').dropna()
df.head()

df['target'] = (df['num'] > 0).astype(int)
df.head()

X = df.drop(columns=['num','target'])
y = df.target

print(f"Rows: {len(X)}, Features: {X.shape[1]}")

Rows: 297, Features: 13


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=52)


In [16]:
# SVM
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)
svm.score(X_test, y_test)

0.6833333333333333

In [17]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=40, random_state=42) # random_state=42 keeps the forest reproducible (it's random without it)
forest.fit(X_train, y_train)
forest.score(X_test, y_test)

0.8

Both scores above came from **one** split. Rerun the split with a different `random_state` and they can swing a lot — so a single number isn't trustworthy. The fix is to test on **many** splits and average them. That's cross-validation.

## 2. K-Fold Cross Validation

**K-Fold** splits the data into *k* equal parts ("folds"). It then runs *k* rounds — each round, one fold is the **test** set and the other *k−1* folds are the **training** set. Every row is tested exactly once, and we average the *k* scores.

First, create a `KFold` splitter with 3 folds:

In [23]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=3)
kf

KFold(n_splits=3, random_state=None, shuffle=False)

In [40]:
data = [1,2,3,4,5,6,7,8,9]

for train_index, test_index in kf.split(data):
    print(train_index, test_index)

# for item in kf.split(data):
#     print(item)

[3 4 5 6 7 8] [0 1 2]
[0 1 2 6 7 8] [3 4 5]
[0 1 2 3 4 5] [6 7 8]


In [41]:
# Manually run 3-fold cross-validation for both models
svm_scores = []
rf_scores = []

for train_index, test_index in kf.split(X, y):
    # use .iloc for positional row selection (X is a DataFrame, so plain X[idx] would look for columns)
    X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    svm = SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))

    rf = RandomForestClassifier(n_estimators=40, random_state=42)
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))

In [45]:
print(svm_scores)
print("Average SVM score: ", np.mean(svm_scores))

[0.6161616161616161, 0.6565656565656566, 0.6060606060606061]
Average SVM score:  0.6262626262626263


In [46]:
print(rf_scores)
print("Average RF score: ", np.mean(rf_scores))

[0.8383838383838383, 0.8080808080808081, 0.797979797979798]
Average RF score:  0.8148148148148149


The average is a single, trustworthy score for each model — and the Random Forest (~0.81) clearly beats the SVM (~0.63) across all folds, not just on one lucky split.

## 3. `cross_val_score` — the shortcut

That manual loop is so common that scikit-learn wraps it in a **single line**. You pass the model, `X`, `y`, and a `cv` strategy, and it returns the array of per-fold scores. Handing it our **same `kf`** reproduces the manual numbers exactly.

In [48]:
from sklearn.model_selection import cross_val_score

cross_val_score(SVC(), X, y, cv=kf)

array([0.61616162, 0.65656566, 0.60606061])

In [51]:
cross_val_score(RandomForestClassifier(n_estimators=40, random_state=42),X, y, cv=kf)

array([0.83838384, 0.80808081, 0.7979798 ])

## 4. Stratified K Fold

`StratifiedKFold` works just like `KFold`, but it keeps the **same class balance** in every fold. Because it needs the class labels to do that, you must pass **both** `X` and `y` to `.split()` (passing only the data is what triggers a `missing argument 'y'` error).

This matters for classification: with plain `KFold`, a fold could accidentally get too few of one class and give a misleading score.

In [52]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=3)

# same idea as the KFold demo -- but StratifiedKFold also needs the labels (y)
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9]
labels  = [0, 0, 0, 0, 0, 1, 1, 1, 1]   # 5 of class 0, 4 of class 1

for train_index, test_index in skf.split(numbers, labels):
    print(train_index, test_index)

[2 3 4 6 7 8] [0 1 5]
[0 1 4 5 7 8] [2 3 6]
[0 1 2 3 5 6] [4 7 8]


In [53]:
# SVM with StratifiedKFold
cross_val_score(SVC(), X, y, cv=skf)

array([0.58585859, 0.66666667, 0.60606061])

In [54]:
# Random Forest with StratifiedKFold
cross_val_score(RandomForestClassifier(n_estimators=40, random_state=42), X, y, cv=skf)

array([0.84848485, 0.7979798 , 0.78787879])

These scores are **identical** to `cross_val_score(model, X, y, cv=3)` — because for a classifier, passing an integer `cv=3` already uses `StratifiedKFold` under the hood. So you rarely need to build one by hand; it's just good to know that's what `cv=3` is doing.